# Finetuning mit Unsloth
- QLoRa is not recommended for finetuning the qwen3.5 model due to highter than normal quantization differences -> LoRa

In [1]:
!nvidia-smi

Sun Jun 21 22:30:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             39W /   70W |    6689MiB /  15360MiB |     38%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Load and prepare the data

In [2]:
import os

# 1. Clone your repo
!git clone https://github.com/klagony/ai_engineering.git /content/repo

# 2. Set working directory to your repo
os.chdir("/content/repo")

# 3. Confirm everything is visible
print(os.listdir("."))

fatal: destination path '/content/repo' already exists and is not an empty directory.
['promting.ipynb', 'unsloth_compiled_cache', '.git', 'README.md', 'results', 'clauses_test.jsonl', 'clauses_train.jsonl', 'requirements.txt', 'utils']


In [3]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files="clauses_train.jsonl", split="train")
test_ds = load_dataset("json", data_files="clauses_test.jsonl", split="train")
print("Loaded dataset with {} training samples and {} test samples.".format(len(train_ds), len(test_ds)))

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded dataset with 800 training samples and 160 test samples.


In [12]:
from datasets import Dataset
SYSTEM_PROMT = "You are a legal assistant that classifies legal clauses into specific categories. Always output ONLY ONE category label without any explanation."

def convert_to_conversation(sample):
    return {
        "messages": [
            {"role": "system",  "content": SYSTEM_PROMT},
            {"role": "user",    "content": str(sample["text"])},
            {"role": "assistant", "content": str(sample["label"])},
        ]
    }

converted_dataset = Dataset.from_list([convert_to_conversation(sample) for sample in train_ds])

In [8]:
converted_dataset[0]

{'messages': [{'role': 'system',
   'content': 'You are a legal assistant that classifies legal clauses into specific categories. Always output ONLY ONE category label without any explanation.'},
  {'role': 'user',
   'content': 'In the event of termination by either party in accordance with any of the provisions of this Agreement, neither party shall be liable to the other, because of such termination, for compensation, reimbursement or damages on account of the loss of prospective profits or anticipated sales or on account of expenditures, inventory, investments, leases or commitments in connection with the business or goodwill of E.piphany or HSNS.'},
  {'role': 'assistant', 'content': 'liability_cap'}]}

## Finetuning

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

 
def finetune_model(model_name, train_ds, r, lora_alpha, lora_dropout, target_modules, lr):

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name, 
        load_in_4bit = False) #enables LoRA 16-bit fine-tuning 
        
    model = FastLanguageModel.get_peft_model(
        model = model,
        r = r,
        lora_alpha = lora_alpha,
        lora_dropout = lora_dropout,
        target_modules = target_modules,
        random_seed = 26) 
    
    FastLanguageModel.for_training(model)

    def render_conversation(sample):
        return {
            "text": tokenizer.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=False  # Qwen3-specific
            )
        }

    rendered_ds = train_ds.map(render_conversation).remove_columns("messages") 

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = rendered_ds,
        dataset_text_field = "text",
        args = SFTConfig(
            per_device_train_batch_size = 2,#4?
            gradient_accumulation_steps = 2,
            warmup_steps = 5,
            num_train_epochs = 1,
            learning_rate = lr,
            logging_steps = 10,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            output_dir = "./results"))

    trainer_stats = trainer.train()
    return model, tokenizer, trainer_stats

### Experiment 1

In [18]:
MODEL_NAME = "unsloth/Qwen3.5-0.8B"

model_r4, tokenizer, trainer_stats = finetune_model(model_name = MODEL_NAME,
                          train_ds = converted_dataset,
                          r = 4,
                          lora_alpha = 8,
                          lora_dropout = 0.1,
                          target_modules = ["q_proj", "k_proj"],#, "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                          lr = 1e-4)

==((====))==  Unsloth 2026.6.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/800 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 159,744 of 853,145,664 (0.02% trained)


Step,Training Loss
10,2.771817
20,2.739614
30,2.614917
40,2.286697
50,2.337948
60,2.096236
70,1.967564
80,2.034628
90,1.894292
100,1.705539


## Interference and evaluation